In [4]:
# This script takes the predicted loads as input (Step 1 from 2-step approach)
# The power predictions from the models (tgt, Sarima, Xgboost, naive) are used as active Power
# The q_mvar var is added by using the case and bus specific scaling factor.
# Output: One csv for each model prediction
# Next step: input the csv files into datakit to retrieve the OPF solution for Step 2 of 2-step approach 

In [5]:
import os
import numpy as np
import pandas as pd
import importlib

#! CONFIG
#! If i want to do case118 i have to setup a new, datakit specific venv, as pypower needs numpy <2 and graphkit needs numpy >2.
CASE = 118                   # set IEEE case 
PARQUET = "../data/data_in/case118_ieee_horizon1.parquet"
OUT_ROOT = "../data/precomputed_profiles/"
ID_COL = "load_scenario_idx"
FIRST_SCENARIO = 22338 #!TODO HACKY -> FIX!

In [6]:
# cell 2 - helper: load IEEE base P/Q via matpowercaseframes
from matpowercaseframes import CaseFrames

def get_ieee_base(case_n: int):
    """Return (P_base, Q_base) arrays for any Matpower case."""
    cf = CaseFrames(f"case{case_n}")
    P_base = cf.bus['PD'].to_numpy(dtype=float)
    Q_base = cf.bus['QD'].to_numpy(dtype=float)
    return P_base, Q_base


In [7]:
# cell 3 - load parquet, validate, derive ratio
if not os.path.exists(PARQUET):
    raise FileNotFoundError(PARQUET)

df = pd.read_parquet(PARQUET)

id_col = ID_COL


# get base P/Q and compute Q/P ratio 
P_base, Q_base = get_ieee_base(CASE)
P_base = np.asarray(P_base, dtype=float)
Q_base = np.asarray(Q_base, dtype=float)
ratio = np.zeros_like(P_base, dtype=float)
mask = P_base != 0.0
ratio[mask] = Q_base[mask] / P_base[mask]

FileNotFoundError: Can't find data at f:\studium\MA_Code\gridfm-datakit\exp1\generate_opf_inputs\case118

In [ ]:
# cell 4 - transform & write one CSV per model column
exclude = {id_col, "bus_id", "horizon_step"}
model_cols = [c for c in df.columns if c not in exclude]

out_dir = os.path.join(OUT_ROOT, f"case{CASE}_ieee")
os.makedirs(out_dir, exist_ok=True)

bus_idx = df["bus_id"].to_numpy(dtype=int)
valid_mask = (bus_idx >= 0) & (bus_idx < len(ratio))
bus_idx_clamped = np.where(valid_mask, bus_idx, -1)

for col in model_cols:
    p_arr = pd.to_numeric(df[col], errors="coerce").fillna(0.0).to_numpy(dtype=float)
    q_arr = np.where(bus_idx_clamped >= 0, p_arr * ratio[bus_idx_clamped], 0.0)
    out = pd.DataFrame({
        "load_scenario": df[id_col].astype(int)-FIRST_SCENARIO, #! Temporary Hack to make datakit work
        "load_scenario_idx": df[id_col].astype(int),
        "load": df["bus_id"].astype(int),
        "p_mw": p_arr,
        "q_mvar": q_arr
    })
    out_path = os.path.join(out_dir, f"{col}.csv")
    out.to_csv(out_path, index=False)
    print("wrote", out_path)

wrote ../data/precomputed_profiles/case14_ieee\true.csv
wrote ../data/precomputed_profiles/case14_ieee\xgb.csv
wrote ../data/precomputed_profiles/case14_ieee\snaive.csv
wrote ../data/precomputed_profiles/case14_ieee\tgt.csv
wrote ../data/precomputed_profiles/case14_ieee\sarima.csv


In [ ]:
df["load_scenario_idx"].nunique()

3942

In [ ]:
df2=pd.read_parquet("../data/data_out/sarima/case14_ieee/raw/bus_data.parquet")

In [ ]:
df2.head()

,scenario,load_scenario_idx,bus,Pd,Qd,Pg,Qg,Vm,Va,PQ,PV,REF,vn_kv,min_vm_pu,max_vm_pu,GS,BS,scenario_partition
0,0,0.0,0,-3.741941e-07,-0.000000,1.996504e+02,0.000153,1.060000,0.000000,0.0,0.0,1.0,1.0,0.94,1.06,0.0,0.0,0
1,0,0.0,1,1.863006e+01,10.903308,-8.586644e-07,20.233989,1.038762,-4.318830,0.0,1.0,0.0,1.0,0.94,1.06,0.0,0.0,0
2,0,0.0,2,6.257194e+01,12.620668,0.000000e+00,15.744058,1.015603,-9.578420,0.0,1.0,0.0,1.0,0.94,1.06,0.0,0.0,0
3,0,0.0,3,3.638837e+01,-2.968925,0.000000e+00,0.000000,1.018241,-8.080713,1.0,0.0,0.0,1.0,0.94,1.06,0.0,0.0,0
4,0,0.0,4,4.577746e+00,0.963736,0.000000e+00,0.000000,1.019749,-6.911958,1.0,0.0,0.0,1.0,0.94,1.06,0.0,0.0,0
